<a href="https://colab.research.google.com/github/FatemehNMT/Visual-SLAM-Book-Google-Colab/blob/main/ch6_ch9_g2o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/NVlabs/intrinsic3d/blob/master/README.md


in jadidre  bebineshhhhhhhhhhhhhh

**Google Ceres** is a widely used optimization library for least-square problems. In
Ceres, as users, we only need to define the optimization problem to be solved according
to specific steps and then hand it over to the solver for calculation.\
 \
In order to tell Ceres the definition of the problem, we need to do the following
things:


*   **Defines each parameter block**. The parameter block is usually a trivial vector,
but it can also be defined as a particular structure such as quaternion and Lie
algebra in SLAM. If it is a vector, we need to allocate a double array for each
parameter block to store the variable's value.
*   Then, define the **calculation method** of the residual block. The residual block is usually associated with several parameter blocks, performs some custom
calculations on them, and then returns the residual value. After that, Ceres
will sum the squares residuals, which is used as the overall objective function.
*   In the **residual blocks**, we also need to define the Jacobian calculation method. In Ceres, we can use the “automatic derivative” function or manually specify the Jacobian calculation process. If you want to use automatic derivation, then the residual block must be written in a specific way: the residual calculation should be implemented as a bracketed operator with a template. We will illustrate this point through an example.
*   Finally, add all the **parameter blocks** and **residual blocks** to Ceres's Problem object and call the Solve function to solve it. Before solving, we can pass some configuration information, such as the number of iterations, termination conditions, etc., or use the default configuration.

# **Chapter 5: Nonlinear Optimization**

**Chapter Reference:** https://github.com/gaoxiang12/slambook2/tree/master/ch6

## **Mount**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pwd

/content


## **Install Eigen**

In [ ]:
!git clone https://gitlab.com/libeigen/eigen.git

Cloning into 'eigen'...
remote: Enumerating objects: 128592, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 128592 (delta 100), reused 168 (delta 91), pack-reused 128404 (from 1)
Receiving objects: 100% (128592/128592), 106.53 MiB | 25.52 MiB/s, done.
Resolving deltas: 100% (106735/106735), done.


In [ ]:
%cd eigen

/content/eigen


In [ ]:
!mkdir build
%cd build

/content/eigen/build


In [ ]:
!cmake ..

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- The Fortran compiler identification is GNU 11.4.0
-- Performing Test standard_math_library_linked_to_automatically
-- Performing Test standard_math_library_linked_to_automatically - Success
-- Standard libraries to link to explicitly: none
-- Performing Test COMPILER_SUPPORT_WERROR
-- Performing Test COMPILER_SUPPORT_WERROR - Success
-- Performing Test COMPILER_SUPPORT_pedantic
-- Performing Test COMPILER_SUPPORT_pedantic - Success
-- Performing Test COMPILER_SUPPORT_Wall
-- Performing T

In [ ]:
!sudo make install

[  0%] Built target eigen_blas_static
[ 25%] Built target eigen_blas
[ 50%] Built target eigen_lapack_static
[100%] Built target eigen_lapack
Install the project...
-- Install configuration: "Release"
-- Up-to-date: /usr/local/include/eigen3/signature_of_eigen3_matrix_library
-- Up-to-date: /usr/local/share/pkgconfig/eigen3.pc
-- Up-to-date: /usr/local/include/eigen3/Eigen
-- Up-to-date: /usr/local/include/eigen3/Eigen/Geometry
-- Up-to-date: /usr/local/include/eigen3/Eigen/MetisSupport
-- Up-to-date: /usr/local/include/eigen3/Eigen/src
-- Up-to-date: /usr/local/include/eigen3/Eigen/src/Geometry
-- Up-to-date: /usr/local/include/eigen3/Eigen/src/Geometry/Translation.h
-- Up-to-date: /usr/local/include/eigen3/Eigen/src/Geometry/Homogeneous.h
-- Up-to-date: /usr/local/include/eigen3/Eigen/src/Geometry/Transform.h
-- Up-to-date: /usr/local/include/eigen3/Eigen/src/Geometry/EulerAngles.h
-- Up-to-date: /usr/local/include/eigen3/Eigen/src/Geometry/InternalHeaderCheck.h
-- Up-to-date: /usr/l

## **Install Ceres**

**TEST and Download ceres-solver-2.1.0.tar:**

https://stackoverflow.com/questions/72368717/how-to-download-ceres-solver-2-1-0-tar-gz-file-to-install-ceres-solver-in-ubun

There are **three** files of

SnavelyReprojectionError.h,

common.h,

common.cpp

that must be run before running the main part of **bundle_adjustment_ceres.cpp**

https://github.com/ceres-solver/ceres-solver

http://ceres-solver.org/installation.html

In [ ]:
%cd /content

/content


In [ ]:
!mkdir Ceres
%cd Ceres

/content/Ceres


In [ ]:
# google-glog + gflags
!sudo apt-get install libgoogle-glog-dev libgflags-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libgflags2.2 libgoogle-glog0v5 libunwind-dev
The following NEW packages will be installed:
  libgflags-dev libgflags2.2 libgoogle-glog-dev libgoogle-glog0v5
  libunwind-dev
0 upgraded, 5 newly installed, 0 to remove and 35 not upgraded.
Need to get 2,207 kB of archives.
After this operation, 7,675 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgflags2.2 amd64 2.2.2-2 [78.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgflags-dev amd64 2.2.2-2 [93.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgoogle-glog0v5 amd64 0.5.0+really0.4.0-2 [60.3 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libunwind-dev amd64 1.3.2-2build2.1 [1,883 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgoogle-glog-dev amd64 0.

In [ ]:
# Use ATLAS for BLAS & LAPACK
!sudo apt-get install libatlas-base-dev


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libatlas-base-dev is already the newest version (3.10.3-12ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
# Eigen3
!sudo apt-get install libeigen3-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  libeigen3-doc libmpfrc++-dev
The following NEW packages will be installed:
  libeigen3-dev
0 upgraded, 1 newly installed, 0 to remove and 35 not upgraded.
Need to get 1,056 kB of archives.
After this operation, 9,081 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libeigen3-dev all 3.4.0-2ubuntu2 [1,056 kB]
Fetched 1,056 kB in 0s (7,669 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected p

In [ ]:
# SuiteSparse (optional)
!sudo apt-get install libsuitesparse-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libamd2 libbtf1 libcamd2 libccolamd2 libcholmod3 libcolamd2 libcxsparse3
  libgraphblas-dev libgraphblas6 libklu1 libldl2 libmetis5 libmongoose2
  librbio2 libsliplu1 libspqr2 libsuitesparseconfig5 libumfpack5
The following NEW packages will be installed:
  libamd2 libbtf1 libcamd2 libccolamd2 libcholmod3 libcolamd2 libcxsparse3
  libgraphblas-dev libgraphblas6 libklu1 libldl2 libmetis5 libmongoose2
  librbio2 libsliplu1 libspqr2 libsuitesparse-dev libsuitesparseconfig5
  libumfpack5
0 upgraded, 19 newly installed, 0 to remove and 35 not upgraded.
Need to get 22.4 MB of archives.
After this operation, 169 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsuitesparseconfig5 amd64 1:5.10.1+dfsg-4build1 [10.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libamd2 amd64 1:5

In [ ]:
!sudo apt-get install liblapack-dev libsuitesparse-dev libcxsparse3 libgflags-dev libgoogle-glog-dev libgtest-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
liblapack-dev is already the newest version (3.10.0-2ubuntu1).
libcxsparse3 is already the newest version (1:5.10.1+dfsg-4build1).
libcxsparse3 set to manually installed.
libgflags-dev is already the newest version (2.2.2-2).
libgoogle-glog-dev is already the newest version (0.5.0+really0.4.0-2).
libsuitesparse-dev is already the newest version (1:5.10.1+dfsg-4build1).
The following additional packages will be installed:
  googletest
The following NEW packages will be installed:
  googletest libgtest-dev
0 upgraded, 2 newly installed, 0 to remove and 35 not upgraded.
Need to get 792 kB of archives.
After this operation, 5,156 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 googletest all 1.11.0-3 [541 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgtest-dev amd64 1.11.0-3 [250 kB]
Fetched 792 kB in 1s (1,116 kB/s)
deb

In [ ]:
!git clone https://github.com/ceres-solver/ceres-solver.git

Cloning into 'ceres-solver'...
remote: Enumerating objects: 23130, done.
remote: Counting objects: 100% (518/518), done.
remote: Compressing objects: 100% (343/343), done.
remote: Total 23130 (delta 295), reused 176 (delta 173), pack-reused 22612 (from 3)
Receiving objects: 100% (23130/23130), 18.03 MiB | 16.11 MiB/s, done.
Resolving deltas: 100% (16089/16089), done.


In [ ]:
!sudo apt-get install libceres-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libceres2
The following NEW packages will be installed:
  libceres-dev libceres2
0 upgraded, 2 newly installed, 0 to remove and 35 not upgraded.
Need to get 2,011 kB of archives.
After this operation, 14.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libceres2 amd64 2.0.0+dfsg1-5 [834 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libceres-dev amd64 2.0.0+dfsg1-5 [1,177 kB]
Fetched 2,011 kB in 0s (13.2 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
de

In [ ]:
!pwd
!ls

/content/Ceres
ceres-solver


In [ ]:
# !unzip '/content/drive/MyDrive/ceres-solver-2.1.0.tar.gz' -d '/ceres-solver/Test_ceres-solver'

!tar -xzf "/content/drive/MyDrive/slambook_data/ch10/ceres-solver-2.1.0.tar.gz" -C "/content/Ceres/ceres-solver"


In [ ]:
%cd ceres-solver


# !mkdir build
# !cd build
# !checkout master
# !cmake ..
# # !cmake ../ceres-solver-2.1.0/
# !make -j32
# !sudo make install
# ORRRRR ...

/content/Ceres/ceres-solver


In [ ]:
!mkdir ceres-bin
%cd ceres-bin

/content/Ceres/ceres-solver/ceres-bin


In [ ]:
!cmake ../ceres-solver-2.1.0/

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test HAVE_BIGOBJ
-- Performing Test HAVE_BIGOBJ - Failed
-- Looking for pow in m
-- Looking for pow in m - found
-- Detected Ceres version: 2.1.0 from /content/Ceres/ceres-solver/ceres-solver-2.1.0/include/ceres/version.h
-- Detected available Ceres threading models: [CXX_THREADS, OPENMP, NO_THREADS]
-- Found Eigen version 3.4.90: /usr/local/share/eigen3/cmake
-- Enabling use of Eigen as a sparse linear algebra library.
CMake Warning (dev) at CMakeLists.txt:230 (find_package):

In [ ]:
!make -j8

[  0%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_2_2.cc.o
[  0%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_2_3.cc.o
[  0%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_2_4.cc.o
[  0%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_2_d.cc.o
[  1%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_3_3.cc.o
[  1%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_3_4.cc.o
[  1%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_3_9.cc.o
[  1%] Building CXX object internal/ceres/CMakeFiles/ceres_internal.dir/generated/partitioned_matrix_view_2_3_6.cc.o
[  1%] Building CXX object internal/ceres/CMakeFiles/ceres_inter

In [ ]:
!make test

Running tests...
Test project /content/Ceres/ceres-solver/ceres-bin
        Start   1: cuda_memcheck_dense_qr_test
Could not find executable cuda-memcheck
Looked in the following places:
cuda-memcheck
cuda-memcheck
Release/cuda-memcheck
Release/cuda-memcheck
Debug/cuda-memcheck
Debug/cuda-memcheck
MinSizeRel/cuda-memcheck
MinSizeRel/cuda-memcheck
RelWithDebInfo/cuda-memcheck
RelWithDebInfo/cuda-memcheck
Deployment/cuda-memcheck
Deployment/cuda-memcheck
Development/cuda-memcheck
Development/cuda-memcheck
Unable to find executable: cuda-memcheck
  1/183 Test   #1: cuda_memcheck_dense_qr_test ...................................***Not Run   0.00 sec
        Start   2: cuda_memcheck_dense_cholesky_test
Could not find executable cuda-memcheck
Looked in the following places:
cuda-memcheck
cuda-memcheck
Release/cuda-memcheck
Release/cuda-memcheck
Debug/cuda-memcheck
Debug/cuda-memcheck
MinSizeRel/cuda-memcheck
MinSizeRel/cuda-memcheck
RelWithDebInfo/cuda-memcheck
RelWithDebInfo/cuda-memcheck
D

In [ ]:
# Optionally install Ceres, it can also be exported using CMake which
# allows Ceres to be used without requiring installation, see the documentation
# for the EXPORT_BUILD_DIR option for more information.
!make install

[ 25%] Built target ceres_internal
[ 27%] Built target ceres
[ 28%] Built target gtest
[ 28%] Built target test_util
[ 28%] Built target array_utils_test
[ 28%] Built target array_selector_test
[ 28%] Built target autodiff_test
[ 28%] Built target autodiff_first_order_function_test
[ 29%] Built target autodiff_cost_function_test
[ 29%] Built target autodiff_local_parameterization_test
[ 30%] Built target autodiff_manifold_test
[ 30%] Built target block_jacobi_preconditioner_test
[ 31%] Built target block_random_access_dense_matrix_test
[ 31%] Built target block_random_access_diagonal_matrix_test
[ 31%] Built target block_random_access_sparse_matrix_test
[ 32%] Built target block_sparse_matrix_test
[ 33%] Built target c_api_test
[ 33%] Built target canonical_views_clustering_test
[ 33%] Built target compressed_col_sparse_matrix_utils_test
[ 34%] Built target compressed_row_sparse_matrix_test
[ 34%] Built target concurrent_queue_test
[ 34%] Built target conditioned_cost_function_test
[ 3

## **Install Sophus**

In [ ]:
%cd /content/

/content


In [ ]:
!git clone https://github.com/strasdat/Sophus.git

Cloning into 'Sophus'...
remote: Enumerating objects: 23331, done.
remote: Counting objects: 100% (387/387), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 23331 (delta 257), reused 230 (delta 230), pack-reused 22944 (from 3)
Receiving objects: 100% (23331/23331), 261.55 MiB | 22.54 MiB/s, done.
Resolving deltas: 100% (15935/15935), done.


In [ ]:
%cd Sophus/

/content/Sophus


In [ ]:
!git clone https://github.com/Microsoft/vcpkg.git
%cd vcpkg

Cloning into 'vcpkg'...
remote: Enumerating objects: 280688, done.
remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 280688 (delta 176), reused 110 (delta 110), pack-reused 280444 (from 2)
Receiving objects: 100% (280688/280688), 87.30 MiB | 23.15 MiB/s, done.
Resolving deltas: 100% (187212/187212), done.
/content/Sophus/vcpkg


In [ ]:
!./bootstrap-vcpkg.sh

vcpkg package management program version 2025-07-16-d6c019e723df46cb8c36360e4174b111455567d3

See LICENSE.txt for license information.
Telemetry
---------
vcpkg collects usage data in order to help us improve your experience.
The data collected by Microsoft is anonymous.
You can opt-out of telemetry by re-running the bootstrap-vcpkg script with -disableMetrics,
passing --disable-metrics to vcpkg on the command line,
or by setting the VCPKG_DISABLE_METRICS environment variable.

Read more about vcpkg telemetry at docs/about/privacy.md


In [ ]:
!./vcpkg integrate install

Applied user-wide integration for this vcpkg root.
CMake projects should use: "-DCMAKE_TOOLCHAIN_FILE=/content/Sophus/vcpkg/scripts/buildsystems/vcpkg.cmake"


In [ ]:
!./vcpkg install sophus

Computing installation plan...
The following packages will be built and installed:
  * eigen3:x64-linux@3.4.0#5
    sophus:x64-linux@1.24.6-r1
  * vcpkg-cmake:x64-linux@2024-04-23
  * vcpkg-cmake-config:x64-linux@2024-05-23
Additional packages (*) will be modified to complete this operation.
Detecting compiler hash for triplet x64-linux...
Compiler found: /usr/bin/c++
Restored 0 package(s) from /root/.cache/vcpkg/archives in 32.6 us. Use --debug to see more details.
Installing 1/4 vcpkg-cmake:x64-linux@2024-04-23...
Building vcpkg-cmake:x64-linux@2024-04-23...
-- Installing: /content/Sophus/vcpkg/packages/vcpkg-cmake_x64-linux/share/vcpkg-cmake/vcpkg_cmake_configure.cmake
-- Installing: /content/Sophus/vcpkg/packages/vcpkg-cmake_x64-linux/share/vcpkg-cmake/vcpkg_cmake_build.cmake
-- Installing: /content/Sophus/vcpkg/packages/vcpkg-cmake_x64-linux/share/vcpkg-cmake/vcpkg_cmake_install.cmake
-- Installing: /content/Sophus/vcpkg/packages/vcpkg-cmake_x64-linux/share/vcpkg-cmake/vcpkg-port-

In [ ]:
!pwd

/content/Sophus/vcpkg


In [ ]:
%cd /content

/content


In [ ]:
# go to the content folder
!mkdir MyExample
%cd MyExample

/content/MyExample


## **Install g2o**

In [ ]:
%cd /content/

/content


### **3rdparty**

https://github.com/gaoxiang12/slambook2/blob/master/ch7/pose_estimation_3d2d.cpp

https://github.com/RainerKuemmerle/g2o/tree/9b41a4ea5ade8e1250b9c1b279f3a9c098811b5a

In [ ]:
# %cd /usr/include
%cd /content/

/content


In [ ]:
!sudo apt-get install freeglut3 freeglut3-dev libeigen3-dev qtdeclarative5-dev qt5-qmake

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libeigen3-dev is already the newest version (3.4.0-2ubuntu2).
The following additional packages will be installed:
  libegl-dev libevdev2 libgl-dev libgl1-mesa-dev libgles-dev libgles1
  libglu1-mesa libglu1-mesa-dev libglvnd-core-dev libglvnd-dev libglx-dev
  libgudev-1.0-0 libinput-bin libinput10 libmd4c0 libmtdev1 libopengl-dev
  libqt5concurrent5 libqt5core5a libqt5dbus5 libqt5gui5 libqt5network5
  libqt5opengl5 libqt5opengl5-dev libqt5printsupport5 libqt5qml5
  libqt5qmlmodels5 libqt5qmlworkerscript5 libqt5quick5 libqt5quickparticles5
  libqt5quickshapes5 libqt5quicktest5 libqt5quickwidgets5 libqt5sql5
  libqt5sql5-sqlite libqt5svg5 libqt5test5 libqt5widgets5 libqt5xml5
  libvulkan-dev libvulkan1 libwacom-bin libwacom-common libwacom9
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxkbcommon-x11-0 libxt-dev

In [ ]:
!sudo apt-get install libsuitesparse-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsuitesparse-dev is already the newest version (1:5.10.1+dfsg-4build1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
!sudo apt-get install libqglviewer-dev-qt5

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libqglviewer-headers libqglviewer2-qt5 libxmu-dev libxmu-headers
The following NEW packages will be installed:
  libqglviewer-dev-qt5 libqglviewer-headers libqglviewer2-qt5 libxmu-dev
  libxmu-headers
0 upgraded, 5 newly installed, 0 to remove and 35 not upgraded.
Need to get 361 kB of archives.
After this operation, 1,452 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libqglviewer2-qt5 amd64 2.6.3+dfsg2-9 [199 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libqglviewer-headers all 2.6.3+dfsg2-9 [49.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libxmu-headers all 2:1.1.3-3 [54.1 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 libxmu-dev amd64 2:1.1.3-3 [54.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libqglviewer-dev-

In [ ]:
!sudo apt-get install libsuitesparse-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsuitesparse-dev is already the newest version (1:5.10.1+dfsg-4build1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
!git clone https://github.com/RainerKuemmerle/g2o.git

Cloning into 'g2o'...
remote: Enumerating objects: 26137, done.
remote: Counting objects: 100% (5663/5663), done.
remote: Compressing objects: 100% (1108/1108), done.
remote: Total 26137 (delta 5002), reused 4603 (delta 4554), pack-reused 20474 (from 3)
Receiving objects: 100% (26137/26137), 10.99 MiB | 17.93 MiB/s, done.
Resolving deltas: 100% (19334/19334), done.


In [ ]:
%cd g2o
!mkdir build
%cd build

/usr/include/g2o
/usr/include/g2o/build


In [ ]:
!cmake ..

-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Compiling on Unix
-- Looking for pthread.h
-- Looking for pthread.h - found
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Found AMD headers in: /usr/include/suitesparse
-- Found AMD library: /usr/lib/x86_64-linux-gnu/libamd.so
-- Found CAMD headers in: /usr/include/suitesparse
-- Found CAMD library: /usr/lib/x86_64-linux-gnu/libcamd.so
-- Found CCOLAMD headers in: /usr/include/suitesparse
-- Found CCOLAMD libra

In [ ]:
!make

[  0%] Building CXX object g2o/EXTERNAL/freeglut/CMakeFiles/freeglut_minimal.dir/freeglut_font.cpp.o
[  0%] Building CXX object g2o/EXTERNAL/freeglut/CMakeFiles/freeglut_minimal.dir/freeglut_stroke_mono_roman.cpp.o
[  1%] Building CXX object g2o/EXTERNAL/freeglut/CMakeFiles/freeglut_minimal.dir/freeglut_stroke_roman.cpp.o
[  1%] Linking CXX shared library ../../../lib/libg2o_ext_freeglut_minimal.so
[  1%] Built target freeglut_minimal
[  1%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/timeutil.cpp.o
[  2%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/command_args.cpp.o
[  2%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/sparse_helper.cpp.o
[  2%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/filesys_tools.cpp.o
[  3%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/string_tools.cpp.o
[  3%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/property.cpp.o
[  3%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/sampler.cpp.o
[  4%] Building CXX object g2o/st

In [ ]:
!make install

Consolidate compiler generated dependencies of target freeglut_minimal
[  1%] Built target freeglut_minimal
Consolidate compiler generated dependencies of target stuff
[  4%] Built target stuff
Consolidate compiler generated dependencies of target opengl_helper
[  5%] Built target opengl_helper
Consolidate compiler generated dependencies of target core
[ 13%] Built target core
Consolidate compiler generated dependencies of target g2o_cli_library
[ 15%] Built target g2o_cli_library
Consolidate compiler generated dependencies of target g2o_cli_application
[ 15%] Built target g2o_cli_application
Consolidate compiler generated dependencies of target types_slam3d
[ 21%] Built target types_slam3d
Consolidate compiler generated dependencies of target types_slam3d_addons
[ 25%] Built target types_slam3d_addons
Consolidate compiler generated dependencies of target types_slam2d
[ 30%] Built target types_slam2d
Consolidate compiler generated dependencies of target types_slam2d_addons
[ 34%] Built

### **(OR) Main**

cmake cant find the eigen3 package


https://askubuntu.com/questions/1265526/cmake-cant-find-the-eigen3-package

In [ ]:
%cd /content

/content


In [ ]:
!git clone https://github.com/RainerKuemmerle/g2o.git

Cloning into 'g2o'...
remote: Enumerating objects: 26137, done.
remote: Counting objects: 100% (5674/5674), done.
remote: Compressing objects: 100% (1107/1107), done.
remote: Total 26137 (delta 5016), reused 4615 (delta 4566), pack-reused 20463 (from 3)
Receiving objects: 100% (26137/26137), 10.99 MiB | 22.06 MiB/s, done.
Resolving deltas: 100% (19339/19339), done.


In [ ]:
%cd /content/g2o

/content/g2o


In [ ]:
!sudo apt-get install libmetis-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libmetis-dev
0 upgraded, 1 newly installed, 0 to remove and 35 not upgraded.
Need to get 5,818 B of archives.
After this operation, 27.6 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmetis-dev amd64 5.1.0.dfsg-7build2 [5,818 B]
Fetched 5,818 B in 0s (32.2 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package libmetis-dev.
(Reading database ... 127579 fi

http://luohanjie.com/2018-08-09/installing-g2o-on-ubuntu.html

In [ ]:
!sudo apt update

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,267 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,157 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [4,964 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,762 kB]
Get:13 http://archive.ubuntu.com/ubuntu ja

In [ ]:
!sudo apt-get install freeglut3 freeglut3-dev libeigen3-dev qtdeclarative5-dev qt5-qmake

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libeigen3-dev is already the newest version (3.4.0-2ubuntu2).
The following additional packages will be installed:
  libegl-dev libevdev2 libgl-dev libgl1-mesa-dev libgles-dev libgles1
  libglu1-mesa libglu1-mesa-dev libglvnd-core-dev libglvnd-dev libglx-dev
  libgudev-1.0-0 libinput-bin libinput10 libmd4c0 libmtdev1 libopengl-dev
  libqt5concurrent5 libqt5core5a libqt5dbus5 libqt5gui5 libqt5network5
  libqt5opengl5 libqt5opengl5-dev libqt5printsupport5 libqt5qml5
  libqt5qmlmodels5 libqt5qmlworkerscript5 libqt5quick5 libqt5quickparticles5
  libqt5quickshapes5 libqt5quicktest5 libqt5quickwidgets5 libqt5sql5
  libqt5sql5-sqlite libqt5svg5 libqt5test5 libqt5widgets5 libqt5xml5
  libvulkan-dev libvulkan1 libwacom-bin libwacom-common libwacom9
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxkbcommon-x11-0 libxt-dev

In [ ]:
!sudo apt-add-repository universe

Adding component(s) 'universe' to all repositories.
Press [ENTER] to continue or Ctrl-c to cancel.
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (s

https://blog.csdn.net/Joweay/article/details/107791083

In [ ]:
!sudo apt-get install libcholmod3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libcholmod3 is already the newest version (1:5.10.1+dfsg-4build1).
libcholmod3 set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


https://blog.csdn.net/Joweay/article/details/107791083

In [ ]:
!sudo apt-get install libsuitesparse-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsuitesparse-dev is already the newest version (1:5.10.1+dfsg-4build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
!sudo add-apt-repository ppa:ubuntuhandbook1/ppa
!sudo apt-get update

Repository: 'deb https://ppa.launchpadcontent.net/ubuntuhandbook1/ppa/ubuntu/ jammy main'
Description:
Qt4-x11 packages that are no longer officially supported! Use it at your own risk!
More info: https://launchpad.net/~ubuntuhandbook1/+archive/ubuntu/ppa
Adding repository.
Press [ENTER] to continue or Ctrl-c to cancel.
Adding deb entry to /etc/apt/sources.list.d/ubuntuhandbook1-ubuntu-ppa-jammy.list
Adding disabled deb-src entry to /etc/apt/sources.list.d/ubuntuhandbook1-ubuntu-ppa-jammy.list
Adding key to /etc/apt/trusted.gpg.d/ubuntuhandbook1-ubuntu-ppa.gpg with fingerprint F4E48910A020E77056748B745738AE8480447DDF
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://developer.download.nvidia.com/compute/

In [ ]:
!sudo apt-get install libqglviewer-dev-qt5

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libqglviewer-headers libqglviewer2-qt5 libxmu-dev libxmu-headers
The following NEW packages will be installed:
  libqglviewer-dev-qt5 libqglviewer-headers libqglviewer2-qt5 libxmu-dev
  libxmu-headers
0 upgraded, 5 newly installed, 0 to remove and 37 not upgraded.
Need to get 361 kB of archives.
After this operation, 1,452 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libqglviewer2-qt5 amd64 2.6.3+dfsg2-9 [199 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libqglviewer-headers all 2.6.3+dfsg2-9 [49.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libxmu-headers all 2:1.1.3-3 [54.1 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 libxmu-dev amd64 2:1.1.3-3 [54.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libqglviewer-dev-

In [ ]:
!sudo apt-get install libglewmx-dev glew-utils libqt4-dev  libsuitesparse-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsuitesparse-dev is already the newest version (1:5.10.1+dfsg-4build1).
The following additional packages will be installed:
  libaudio2 libglewmx1.13 libmng2 libqt4-dbus libqt4-declarative
  libqt4-designer libqt4-dev-bin libqt4-help libqt4-network libqt4-opengl
  libqt4-opengl-dev libqt4-qt3support libqt4-script libqt4-scripttools
  libqt4-sql libqt4-sql-mysql libqt4-svg libqt4-test libqt4-xml
  libqt4-xmlpatterns libqtcore4 libqtdbus4 libqtgui4 qdbus qt4-linguist-tools
  qt4-qmake qtcore4-l10n
Suggested packages:
  nas libqt4-declarative-folderlistmodel libqt4-declarative-gestures
  libqt4-declarative-particles libqt4-declarative-shaders qt4-qmlviewer
  firebird-dev qt4-dev-tools qt4-doc libicu57 qt4-qtconfig
Recommended packages:
  qt-at-spi
The following NEW packages will be installed:
  glew-utils libaudio2 libglewmx-dev libglewmx1.13 libmng2 libqt4-dbus
  libqt4-declarative libqt4-

In [ ]:
!sudo apt install libglew-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following package was automatically installed and is no longer required:
  libglewmx1.13
Use 'sudo apt autoremove' to remove it.
The following packages will be REMOVED:
  libglewmx-dev
The following NEW packages will be installed:
  libglew-dev
0 upgraded, 1 newly installed, 1 to remove and 37 not upgraded.
Need to get 287 kB of archives.
After this operation, 1,526 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libglew-dev amd64 2.2.0-4 [287 kB]
Fetched 287 kB in 0s (2,851 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)

In [ ]:
%cd /content/g2o
!mkdir build
%cd build

/content/g2o
/content/g2o/build


In [ ]:
!cmake ..

-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Compiling on Unix
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Found AMD headers in: /usr/include/suitesparse
-- Found AMD library: /usr/lib/x86_64-linux-gnu/libamd.so
-- Found CAMD headers in: /usr/include/suitesparse
-- Found CAMD library: /usr/lib/x86_64-linux-gnu/libcamd.so
-- Found CCOLAMD headers in: /usr/include/suitesparse
-- Found CCOLAMD library: /usr/lib/x86_64-linux-gnu/libccolamd.so
-- Found CHOLMOD

In [ ]:
!make

[  0%] Building CXX object g2o/EXTERNAL/freeglut/CMakeFiles/freeglut_minimal.dir/freeglut_font.cpp.o
[  0%] Building CXX object g2o/EXTERNAL/freeglut/CMakeFiles/freeglut_minimal.dir/freeglut_stroke_mono_roman.cpp.o
[  1%] Building CXX object g2o/EXTERNAL/freeglut/CMakeFiles/freeglut_minimal.dir/freeglut_stroke_roman.cpp.o
[  1%] Linking CXX shared library ../../../lib/libg2o_ext_freeglut_minimal.so
[  1%] Built target freeglut_minimal
[  1%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/timeutil.cpp.o
[  2%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/command_args.cpp.o
[  2%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/sparse_helper.cpp.o
[  2%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/filesys_tools.cpp.o
[  3%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/string_tools.cpp.o
[  3%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/property.cpp.o
[  3%] Building CXX object g2o/stuff/CMakeFiles/stuff.dir/sampler.cpp.o
[  4%] Building CXX object g2o/st

In [ ]:
!make install

[  1%] Built target freeglut_minimal
[  4%] Built target stuff
[  5%] Built target opengl_helper
[ 13%] Built target core
[ 15%] Built target g2o_cli_library
[ 15%] Built target g2o_cli_application
[ 21%] Built target types_slam3d
[ 25%] Built target types_slam3d_addons
[ 30%] Built target types_slam2d
[ 34%] Built target types_slam2d_addons
[ 41%] Built target g2o_simulator_library
[ 42%] Built target g2o_simulator2d_application
[ 42%] Built target g2o_simulator3d_application
[ 47%] Built target viewer_library
[ 47%] Built target g2o_viewer
[ 50%] Built target types_data
[ 52%] Built target types_sclam2d
[ 59%] Built target types_sba
[ 59%] Built target types_icp
[ 60%] Built target types_sim3
[ 61%] Built target solver_pcg
[ 62%] Built target solver_dense
[ 62%] Built target solver_eigen
[ 63%] Built target solver_slam2d_linear
[ 64%] Built target solver_structure_only
[ 65%] Built target csparse_extension
[ 66%] Built target solver_csparse
[ 67%] Built target solver_cholmod
[ 68%] B

### **Test installed g2o**

After installing **g2o** and **eigen** and testing it,


To Test adding code to the main folder of installed g2o:

1. Download one of them (here: curve_fit.cpp), because the version and dependencies and Cmake parts are important and must be corrected.
2. Change CMakeLists.txt file.
3. Run the required codes such as **cmake** and **make**.
4. See the results in the bin folder, like **!./bin/circle_fiit**.

In [ ]:
%cd /content/g2o/build

/content/g2o/build


In [ ]:
!./bin/curve_fit

Target curve
a * exp(-lambda * x) + b
Iterative least squares solution
a      = 1.98063
b      = 0.416872
lambda = 0.203127



## **SLAM Book 2: Curve Fitting with g2o**

**Page 119**\
https://github.com/gaoxiang12/slambook2/blob/master/ch6/g2oCurveFitting.cpp

In [ ]:
%cd /content/

/content


In [ ]:
!rm -r MyG2oExample
!mkdir MyG2oExample
%cd MyG2oExample

rm: cannot remove 'MyG2oExample': No such file or directory
/content/MyG2oExample


In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.22)
project(ch6)

set(CMAKE_BUILD_TYPE "Release")
add_definitions("-DENABLE_SSE")

set(CMAKE_CXX_FLAGS "-O3 -std=c++17")
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_EXTENSIONS OFF)
add_compile_options(-std=c++17)

# OpenCV
find_package(OpenCV REQUIRED)
include_directories(${OpenCV_INCLUDE_DIRS})
if(OpenCV_FOUND)
  MESSAGE( "OpenCV Directory Found.")
  MESSAGE(${OpenCV_INCLUDE_DIRS} )
endif()

# Eigen3
include_directories("/usr/include/eigen3")
find_package(Eigen3 REQUIRED)

if(Eigen3_FOUND)
  MESSAGE( "eigen3 Directory Found.")
endif()

# Eigen
include_directories("/content/eigen")

# g2o
include_directories("/content/g2o")
include_directories("/usr/local/include")

list(APPEND CMAKE_MODULE_PATH "/content/g2o/cmake_modules")
LIST( APPEND CMAKE_MODULE_PATH ${/content/g2o/cmake_modules})

set(CMAKE_PREFIX_PATH "/content/g2o" ${CMAKE_PREFIX_PATH})
# set(CMAKE_PREFIX_PATH "/content/g2o/cmake_modules/FindG2O.cmake" ${CMAKE_PREFIX_PATH})
set(CMAKE_PREFIX_PATH "/content/g2o/cmake_modules" ${CMAKE_PREFIX_PATH})
set(CMAKE_PREFIX_PATH "/content/gdrive/MyDrive/slambook_data/ch6/FindG2O.cmake" ${CMAKE_PREFIX_PATH})

SET( G2O_ROOT /content/g2o )
set(CMAKE_PREFIX_PATH "/usr/local/include/" ${CMAKE_PREFIX_PATH})
FIND_PACKAGE( G2O )
SET( G2O_ROOT "/usr/local/include/" )
SET( G2O_ROOT "/content/g2o" )

set(G2O_LIB_VERSION   "0.2.0" CACHE STRING "g2o library version")
set(G2O_LIB_SOVERSION "0.2"   CACHE STRING "g2o library soversion")
set(G2O_VERSION 2.0.0)

configure_file("/content/g2o/cmake_modules/Config.cmake.in" "${G2O_PROJECT_CONFIG}" @ONLY)
# configure_file("${g2o_SOURCE_DIR}/cmake_modules/Config.cmake.in" "${G2O_PROJECT_CONFIG}" @ONLY)
# list(APPEND CMAKE_MODULE_PATH ${g2o_SOURCE_DIR}/cmake_modules)

FIND_PACKAGE(G2O REQUIRED)
if(G2o_FOUND)
  MESSAGE( "G2o Directory Found.")
  # MESSAGE(${eigen3_INCLUDE_DIRS} )
endif()

# Allow the developer to select if Dynamic or Static libraries are built
option (BUILD_SHARED_LIBS "Build Shared Libraries (preferred and required for the g2o plugin system)" ON)
set (G2O_LIB_TYPE STATIC)
if (BUILD_SHARED_LIBS)
  set (G2O_LIB_TYPE SHARED)
endif()

# For building the CHOLMOD based solvers
option(G2O_USE_CHOLMOD "Build g2o with CHOLMOD support" ON)
find_package(SuiteSparse)
if (G2O_USE_CHOLMOD AND SuiteSparse_CHOLMOD_FOUND)
  message(STATUS "Enable support for Cholmod")
  set(CHOLMOD_FOUND TRUE)
else()
  message(STATUS "Disable support for Cholmod")
  set(CHOLMOD_FOUND FALSE)
endif()

# Options to control the LGPL libraries
option(G2O_USE_LGPL_LIBS "Build libraries which use LGPL code" TRUE)

# If the LGPL libraries are used, check if static or shared libraries are used and
# show a suitable message
if (G2O_USE_LGPL_LIBS)
  if (G2O_LIB_TYPE STREQUAL "STATIC")
    message(STATUS "Building LGPL code as a static library (affects license of the binary)")
  else()
    message(STATUS "Building LGPL code as a shared library")
  endif()
endif()

# Adapter for the legacy LGPL flags. Note there is an inconsistency with the old implementation.
if (BUILD_LGPL_SHARED_LIBS)
  if (G2O_LIB_TYPE STREQUAL "SHARED")
    set(G2O_USE_LGPL_LIBS TRUE)
  else()
    message(FATAL_ERROR "BUILD_LGPL_SHARED_LIBS is set to true but G2O_LIB_TYPE is set to STATIC")
  endif()
endif()

# For building the CSparse based solvers. Note this depends on an LGPL library.
option(G2O_USE_CSPARSE "Build g2o with CSParse support" ON)
find_package(CSparse)
if (${G2O_USE_CSPARSE} AND ${CSPARSE_FOUND} AND ${G2O_USE_LGPL_LIBS})
  message(STATUS "Enable support for CSparse")
else()
  message(STATUS "Disable support for CSparse")
  set(G2O_USE_CSPARSE FALSE)
endif()

# get_property(dirs DIRECTORY ${EIGEN3_INCLUDE_DIR} PROPERTY INCLUDE_DIRECTORIES)
# foreach(dir ${dirs})
#   message(STATUS "dir='${dir}'")
# endforeach()

add_executable(g2oCurveFitting g2oCurveFitting.cpp)
target_link_libraries(g2oCurveFitting ${OpenCV_LIBS} ${G2O_CORE_LIBRARY} ${G2O_STUFF_LIBRARY})

# add_executable(curve_fit curve_fit.cpp)
# set_target_properties(curve_fit PROPERTIES OUTPUT_NAME curve_fit)
# target_link_libraries(curve_fit core solver_dense)

Writing CMakeLists.txt


**g2o Steps:**

1. Define the type of vertices and edges.
2. Build the graph.
3. Select the optimization algorithm.
4. Call g2o to optimize and get the result.

. یه سری ایکس    و   وای   و  تخمین    داریم

. با  ایکسها و تخمینها، گوشه ها و بعد گراف رو تعریف میکنیم

. انتخاب روش بهینه سازی

. اجرا

We **first** declare the g2o graph optimizer and configure the solver and gradient descent method.

**Then**, based on the estimated feature points, we put the pose and spatial points into the graph.

**Finally**, the optimization function is called.


1. I have replaced g2o::make_unique with std::make_unique. Doesn't worked yet.

2. I am installing g2o in the include file to may be found through the CMakeLists.txt

In [ ]:
%%writefile g2oCurveFitting.cpp

#include <iostream>
#include <g2o/core/g2o_core_api.h>
#include <g2o/core/base_vertex.h>
#include <g2o/core/base_unary_edge.h>
#include <g2o/core/block_solver.h>
#include <g2o/core/optimization_algorithm_levenberg.h>
#include <g2o/core/optimization_algorithm_gauss_newton.h>
#include <g2o/core/optimization_algorithm_dogleg.h>
#include <g2o/solvers/dense/linear_solver_dense.h>
#include <Eigen/Core>
#include <opencv2/core/core.hpp>
#include <cmath>
#include <chrono>
// #include <unistd.h>

using namespace std;

// vertex: 3d vector
// Vertices of curve models, template parameters: Optimizing variable dimensions and data types
class CurveFittingVertex : public g2o::BaseVertex<3, Eigen::Vector3d> {
public:
  EIGEN_MAKE_ALIGNED_OPERATOR_NEW

  // reset
  // override the reset function
  virtual void setToOriginImpl() override {
    _estimate << 0, 0, 0;
  }

  // renew
  // override the plus operator, just plain vector addition
  virtual void oplusImpl(const double *update) override {
    _estimate += Eigen::Vector3d(update);
  }

  // Save and load: Leave blank
  // the dummy read/write function
  virtual bool read(istream &in) {}

  virtual bool write(ostream &out) const {}
};

// The entire problem has only one vertex: the parameters a, b, c of the curve model.
// Each noisy data point constitutes an error term, which is the edge of the graph
// optimization. But the edges here are not the same as we usually think. They are
// unary edges, which means that the edges connect only one vertex. Because the
// entire graph has only one vertex.
// (PDF 118 (136/356))

// So:
// 1. only one vertex (parameters of the resulting curve)
// 2. dimension of observation, 1D

// edge: 1D error term, connected to exactly one vertex
// Error model template parameters: observation dimension, type, connection vertex type
class CurveFittingEdge : public g2o::BaseUnaryEdge<1, double, CurveFittingVertex> {
public:
  EIGEN_MAKE_ALIGNED_OPERATOR_NEW

  CurveFittingEdge(double x) : BaseUnaryEdge(), _x(x) {}

  // Calculate curve model error
  virtual void computeError() override {
    const CurveFittingVertex *v = static_cast<const CurveFittingVertex *> (_vertices[0]);
    const Eigen::Vector3d abc = v->estimate();
    _error(0, 0) = _measurement - std::exp(abc(0, 0) * _x * _x + abc(1, 0) * _x + abc(2, 0));
  }

  // Calculate the Jacobian matrix
  virtual void linearizeOplus() override {
    const CurveFittingVertex *v = static_cast<const CurveFittingVertex *> (_vertices[0]);
    const Eigen::Vector3d abc = v->estimate();
    double y = exp(abc[0] * _x * _x + abc[1] * _x + abc[2]);
    _jacobianOplusXi[0] = -_x * _x * y;
    _jacobianOplusXi[1] = -_x * y;
    _jacobianOplusXi[2] = -y;
  }

  virtual bool read(istream &in) {}

  virtual bool write(ostream &out) const {}

public:
  double _x;  // x value, the y value is _measurement
};

int main(int argc, char **argv) {
  double ar = 1.0, br = 2.0, cr = 1.0;         // real parameter values
  double ae = 2.0, be = -1.0, ce = 5.0;        // Estimate parameter values
  int N = 100;                                 // data points
  double w_sigma = 1.0;                        // Noise Sigma value
  double inv_sigma = 1.0 / w_sigma;
  cv::RNG rng;                                 // OpenCV random number generator

  vector<double> x_data, y_data;               // data
  for (int i = 0; i < N; i++) {
    double x = i / 100.0;
    x_data.push_back(x);
    y_data.push_back(exp(ar * x * x + br * x + cr) + rng.gaussian(w_sigma * w_sigma));
  }

// *****************************************************************

  // Build graph optimization, first set up g2o
  typedef g2o::BlockSolver<g2o::BlockSolverTraits<3, 1>> BlockSolverType;
  // The optimization variable dimension of each error term is 3, and the error value dimension is 1
  // pose is 3, landmark is 1

  typedef g2o::LinearSolverDense<BlockSolverType::PoseMatrixType> LinearSolverType; // Linear solver type

  // Gradient descent method, you can choose from GN, LM, DogLeg
  auto solver = new g2o::OptimizationAlgorithmGaussNewton(
    std::make_unique<BlockSolverType>(std::make_unique<LinearSolverType>()));
  g2o::SparseOptimizer optimizer;   // graphical model
  optimizer.setAlgorithm(solver);   // Set up solver
  optimizer.setVerbose(true);       // Turn on debug output

  // Add vertices to the graph
  CurveFittingVertex *v = new CurveFittingVertex();
  v->setEstimate(Eigen::Vector3d(ae, be, ce));
  v->setId(0);
  optimizer.addVertex(v);

  // Add edges to the graph
  // Observations
  for (int i = 0; i < N; i++) {
    CurveFittingEdge *edge = new CurveFittingEdge(x_data[i]);
    edge->setId(i);
    edge->setVertex(0, v);                // Set connected vertices
    edge->setMeasurement(y_data[i]);      // Observed values
    edge->setInformation(Eigen::Matrix<double, 1, 1>::Identity() * 1 / (w_sigma * w_sigma));
    // Information Matrix: Inverse of Covariance Matrix

    optimizer.addEdge(edge);
  }

  // Perform optimization
  cout << "start optimization" << endl;
  chrono::steady_clock::time_point t1 = chrono::steady_clock::now();
  optimizer.initializeOptimization();
  optimizer.optimize(10);
  chrono::steady_clock::time_point t2 = chrono::steady_clock::now();
  chrono::duration<double> time_used = chrono::duration_cast<chrono::duration<double>>(t2 - t1);
  cout << "solve time cost = " << time_used.count() << " seconds. " << endl;

  // Output optimization value
  Eigen::Vector3d abc_estimate = v->estimate();
  cout << "estimated model: " << abc_estimate.transpose() << endl;

  // Eigen::Matrix3f nums;
  // nums << 1,2,3,4,5,6,7,8,9;
  // cout << nums << '\n';
  // cout << "****" << '\n';

  return 0;
}

Writing g2oCurveFitting.cpp


In [ ]:
!rm -r build
!mkdir build
%cd build/

rm: cannot remove 'build': No such file or directory
/content/MyG2oExample/build


In [ ]:
!cmake ..

In [ ]:
!make g2oCurveFitting VERBOSE=1

In [ ]:
! ./g2oCurveFitting

start optimization
iteration= 0	 chi2= 376785.128234	 time= 1.9811e-05	 cumTime= 1.9811e-05	 edges= 100	 schur= 0
iteration= 1	 chi2= 35673.566018	 time= 1.5389e-05	 cumTime= 3.52e-05	 edges= 100	 schur= 0
iteration= 2	 chi2= 2195.012304	 time= 8.462e-06	 cumTime= 4.3662e-05	 edges= 100	 schur= 0
iteration= 3	 chi2= 174.853126	 time= 8.286e-06	 cumTime= 5.1948e-05	 edges= 100	 schur= 0
iteration= 4	 chi2= 102.779695	 time= 8.321e-06	 cumTime= 6.0269e-05	 edges= 100	 schur= 0
iteration= 5	 chi2= 101.937194	 time= 1.4484e-05	 cumTime= 7.4753e-05	 edges= 100	 schur= 0
iteration= 6	 chi2= 101.937020	 time= 8.976e-06	 cumTime= 8.3729e-05	 edges= 100	 schur= 0
iteration= 7	 chi2= 101.937020	 time= 8.333e-06	 cumTime= 9.2062e-05	 edges= 100	 schur= 0
iteration= 8	 chi2= 101.937020	 time= 8.038e-06	 cumTime= 0.0001001	 edges= 100	 schur= 0
iteration= 9	 chi2= 101.937020	 time= 8.3e-06	 cumTime= 0.0001084	 edges= 100	 schur= 0
solve time cost = 0.000851857 seconds. 
estimated model: 0.890912   

## **Bundle Adjustment with g2o**

In [ ]:
%cd /content/BA_Example

/content/BA_Example


In [ ]:
!git clone https://github.com/gaoxiang12/g2o_ba_example.git

Cloning into 'g2o_ba_example'...
remote: Enumerating objects: 28, done.
remote: Total 28 (delta 0), reused 0 (delta 0), pack-reused 28 (from 1)
Receiving objects: 100% (28/28), 789.74 KiB | 5.16 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [ ]:
%cd /content/BA_Example/g2o_ba_example

/content/BA_Example/g2o_ba_example


In [ ]:
%%writefile CMakeLists.txt
# CMake 工程，读者应该熟悉了，我就不一一注释了
cmake_minimum_required( VERSION 3.31 )
project( g2o_ba_example )

set( CMAKE_BUILD_TYPE Release )
set( CMAKE_CXX_FLAGS "-std=c++17 -Wall -O2 -march=native" )

list( APPEND CMAKE_MODULE_PATH ${PROJECT_SOURCE_DIR}/cmake_modules )
include_directories("/content/g2o")
include_directories("/content/g2o/g2o/solvers/cholmod")
include_directories("/content/g2o/g2o/solvers/csparse")
include_directories("/content/g2o/g2o/solvers/dense")
include_directories("/content/g2o/g2o/solvers/eigen")
include_directories("/content/g2o/g2o/core")
include_directories("/usr/local/lib")

include_directories("/usr/include/opencv4/opencv2")


#link_directories(${G2O_PATH}/lib)
link_directories("/usr/local/lib")

SET(G2O_LIBS g2o_cli g2o_ext_freeglut_minimal g2o_simulator g2o_solver_slam2d_linear
        g2o_types_icp g2o_types_slam2d g2o_core g2o_interface g2o_solver_csparse
        g2o_solver_structure_only g2o_types_sba g2o_types_slam3d g2o_csparse_extension
        g2o_opengl_helper g2o_stuff g2o_types_sclam2d g2o_parser
        g2o_solver_dense g2o_solver_pcg g2o_solver_cholmod g2o_types_data g2o_types_sim3 ${SuiteSparse_LIBRARIES}
        cxsparse # 要加上，否则会出错
        )

#find_package( G2O REQUIRED )
find_package( OpenCV REQUIRED )
find_package( Eigen3 REQUIRED )
#find_package( Cholmod )
# Eigen3
include_directories("/usr/include/eigen3")
find_package(Eigen3 REQUIRED)

if(Eigen3_FOUND)
  MESSAGE( "eigen3 Directory Found.")
endif()

# Eigen
include_directories("/content/eigen")
include_directories( ${EIGEN3_INCLUDE_DIR} ${CHOLMOD_INCLUDE_DIR} ${G2O_INCLUDE_DIRS} ${OpenCV_INCLUDE_DIRS} ${CSPARSE_INCLUDE_DIR})

add_executable( ba_example main.cpp )
target_link_libraries( ba_example
    ${OpenCV_LIBS}
    g2o_core g2o_types_slam3d g2o_solver_csparse g2o_stuff g2o_csparse_extension g2o_types_sba
    ${CHOLMOD_LIBRARIES}
    ${OpenCV_LIBS} ${G2O_CORE_LIBRARY} ${G2O_LIBS} ${G2O_STUFF_LIBRARY}  ${G2O_SOLVERS_LIBRARY}
    )



Overwriting CMakeLists.txt


In [ ]:
%%writefile main.cpp
/**
 * BA Example
 * Author: Xiang Gao
 * Date: 2016.3
 * Email: gaoxiang12@mails.tsinghua.edu.cn
 *
 * 在这个程序中，我们读取两张图像，进行特征匹配。然后根据匹配得到的特征，计算相机运动以及特征点的位置。这是一个典型的Bundle Adjustment，我们用g2o进行优化。
 */

// for std
#include <iostream>
// for opencv
#include <opencv2/core/core.hpp>
#include <opencv2/highgui/highgui.hpp>
#include <opencv2/features2d/features2d.hpp>
#include <boost/concept_check.hpp>
// for g2o
#include <g2o/core/sparse_optimizer.h>
#include <g2o/core/block_solver.h>
#include <g2o/core/robust_kernel.h>
#include <g2o/core/robust_kernel_impl.h>
#include <g2o/core/optimization_algorithm_levenberg.h>
#include <g2o/solvers/cholmod/linear_solver_cholmod.h>
#include <g2o/types/slam3d/se3quat.h>
#include <g2o/types/sba/types_six_dof_expmap.h>


using namespace std;

// 寻找两个图像中的对应点，像素坐标系
// 输入：img1, img2 两张图像
// 输出：points1, points2, 两组对应的2D点
int     findCorrespondingPoints( const cv::Mat& img1, const cv::Mat& img2, vector<cv::Point2f>& points1, vector<cv::Point2f>& points2 );

// 相机内参
double cx = 325.5;
double cy = 253.5;
double fx = 518.0;
double fy = 519.0;

int main( int argc, char** argv )
{
    // 调用格式：命令 [第一个图] [第二个图]
    // if (argc != 3)
    // {
    //     cout<<"Usage: ba_example img1, img2"<<endl;
    //     exit(1);
    // }

    // 读取图像
    // cv::Mat img1 = cv::imread( argv[1] );
    // cv::Mat img2 = cv::imread( argv[2] );
    // string image_file_1 = "/content/BA_Example/g2o_ba_example/data/1.png";
    // cv::Mat image;
    // image = cv::imread(image_file);
    cv::Mat img1 = cv::imread("/content/BA_Example/g2o_ba_example/data/1.png");
    cv::Mat img2 = cv::imread("/content/BA_Example/g2o_ba_example/data/2.png");
    // 找到对应点
    vector<cv::Point2f> pts1, pts2;
    if ( findCorrespondingPoints( img1, img2, pts1, pts2 ) == false )
    {
        imwrite("image_1.png", img1);
        imwrite("image_2.png", img2);


        cout<<"匹配点不够！"<<endl;
        return 0;
    }
    cout<<"找到了"<<pts1.size()<<"组对应特征点。"<<endl;
    // 构造g2o中的图
    // 先构造求解器
    g2o::SparseOptimizer    optimizer;
    // 使用Cholmod中的线性方程求解器
    //g2o::BlockSolver_6_3::LinearSolverType* linearSolver = new  g2o::LinearSolverCholmod<g2o::BlockSolver_6_3::PoseMatrixType> ();
    std::unique_ptr<g2o::BlockSolver_6_3::LinearSolverType> linearSolver (new g2o::LinearSolverCholmod<g2o::BlockSolver_6_3::PoseMatrixType>());
    // 6*3 的参数
    //g2o::BlockSolver_6_3* block_solver = new g2o::BlockSolver_6_3( linearSolver );
    std::unique_ptr<g2o::BlockSolver_6_3> solver_ptr (new g2o::BlockSolver_6_3(std::move(linearSolver)));
    g2o::OptimizationAlgorithmLevenberg * algorithm = new g2o::OptimizationAlgorithmLevenberg(std::move(solver_ptr));
    // L-M 下降
    //g2o::OptimizationAlgorithmLevenberg* algorithm = new g2o::OptimizationAlgorithmLevenberg( block_solver );

    optimizer.setAlgorithm( algorithm );
    optimizer.setVerbose( false );

    // 添加节点
    // 两个位姿节点
    for ( int i=0; i<2; i++ )
    {
        g2o::VertexSE3Expmap* v = new g2o::VertexSE3Expmap();
        v->setId(i);
        if ( i == 0)
            v->setFixed( true ); // 第一个点固定为零
        // 预设值为单位Pose，因为我们不知道任何信息
        v->setEstimate( g2o::SE3Quat() );
        optimizer.addVertex( v );
    }
    // 很多个特征点的节点
    // 以第一帧为准
    for ( size_t i=0; i<pts1.size(); i++ )
    {
        g2o::VertexPointXYZ * v = new g2o::VertexPointXYZ ();
        v->setId( 2 + i );
        // 由于深度不知道，只能把深度设置为1了
        double z = 1;
        double x = ( pts1[i].x - cx ) * z / fx;
        double y = ( pts1[i].y - cy ) * z / fy;
        v->setMarginalized(true);
        v->setEstimate( Eigen::Vector3d(x,y,z) );
        optimizer.addVertex( v );
    }

    // 准备相机参数
    g2o::CameraParameters* camera = new g2o::CameraParameters( fx, Eigen::Vector2d(cx, cy), 0 );
    camera->setId(0);
    optimizer.addParameter( camera );

    // 准备边
    // 第一帧
    vector<g2o::EdgeProjectXYZ2UV*> edges;
    for ( size_t i=0; i<pts1.size(); i++ )
    {
        g2o::EdgeProjectXYZ2UV*  edge = new g2o::EdgeProjectXYZ2UV();
        edge->setVertex( 0, dynamic_cast<g2o::VertexPointXYZ *>   (optimizer.vertex(i+2)) );
        edge->setVertex( 1, dynamic_cast<g2o::VertexSE3Expmap*>     (optimizer.vertex(0)) );
        edge->setMeasurement( Eigen::Vector2d(pts1[i].x, pts1[i].y ) );
        edge->setInformation( Eigen::Matrix2d::Identity() );
        edge->setParameterId(0, 0);
        // 核函数
        edge->setRobustKernel( new g2o::RobustKernelHuber() );
        optimizer.addEdge( edge );
        edges.push_back(edge);
    }
    // 第二帧
    for ( size_t i=0; i<pts2.size(); i++ )
    {
        g2o::EdgeProjectXYZ2UV*  edge = new g2o::EdgeProjectXYZ2UV();
        edge->setVertex( 0, dynamic_cast<g2o::VertexPointXYZ *>   (optimizer.vertex(i+2)) );
        edge->setVertex( 1, dynamic_cast<g2o::VertexSE3Expmap*>     (optimizer.vertex(1)) );
        edge->setMeasurement( Eigen::Vector2d(pts2[i].x, pts2[i].y ) );
        edge->setInformation( Eigen::Matrix2d::Identity() );
        edge->setParameterId(0,0);
        // 核函数
        edge->setRobustKernel( new g2o::RobustKernelHuber() );
        optimizer.addEdge( edge );
        edges.push_back(edge);
    }

    cout<<"开始优化"<<endl;
    optimizer.setVerbose(true);
    optimizer.initializeOptimization();
    optimizer.optimize(10);
    cout<<"优化完毕"<<endl;

    //我们比较关心两帧之间的变换矩阵
    g2o::VertexSE3Expmap* v = dynamic_cast<g2o::VertexSE3Expmap*>( optimizer.vertex(1) );
    Eigen::Isometry3d pose = v->estimate();
    cout<<"Pose="<<endl<<pose.matrix()<<endl;

    // 以及所有特征点的位置
    for ( size_t i=0; i<pts1.size(); i++ )
    {
        g2o::VertexPointXYZ * v = dynamic_cast<g2o::VertexPointXYZ *> (optimizer.vertex(i+2));
        cout<<"vertex id "<<i+2<<", pos = ";
        Eigen::Vector3d pos = v->estimate();
        cout<<pos(0)<<","<<pos(1)<<","<<pos(2)<<endl;
    }

    // 估计inlier的个数
    int inliers = 0;
    for ( auto e:edges )
    {
        e->computeError();
        // chi2 就是 error*\Omega*error, 如果这个数很大，说明此边的值与其他边很不相符
        if ( e->chi2() > 1 )
        {
            cout<<"error = "<<e->chi2()<<endl;
        }
        else
        {
            inliers++;
        }
    }

    cout<<"inliers in total points: "<<inliers<<"/"<<pts1.size()+pts2.size()<<endl;
    optimizer.save("ba.g2o");
    return 0;
}

int findCorrespondingPoints (const cv::Mat& img1, const cv::Mat& img2, vector<cv::Point2f>& points1, vector<cv::Point2f>& points2)
{
    // cv::ORB orb;

    // cv::Ptr<cv::ORB> orb = cv::ORB::create();
    cv::Ptr<cv::ORB> orb = cv::ORB::create();

    // cv::Ptr<ORB> orb;
    // orb = ORB::create();

    // orb = cv::Ptr(new cv::ORB())

    //orb* ptr = new ORB();
    //ptr->someFunction();
    //delete ptr;

    std::vector<cv::KeyPoint> kp1, kp2;
    cv::Mat desp1, desp2;

    //orb( img1, cv::Mat(), kp1, desp1 );
    //orb( img2, cv::Mat(), kp2, desp2 );
    orb->detectAndCompute(img1, cv::noArray(), kp1, desp1);
    orb->detectAndCompute(img2, cv::noArray(), kp2, desp2);
    cout<<"分别找到了"<<kp1.size()<<"和"<<kp2.size()<<"个特征点"<<endl;

    cv::Ptr<cv::DescriptorMatcher>  matcher = cv::DescriptorMatcher::create( "BruteForce-Hamming");

    double knn_match_ratio=0.8;
    vector< vector<cv::DMatch> > matches_knn;
    matcher->knnMatch( desp1, desp2, matches_knn, 2 );
    vector< cv::DMatch > matches;
    for ( size_t i=0; i<matches_knn.size(); i++ )
    {
        if (matches_knn[i][0].distance < knn_match_ratio * matches_knn[i][1].distance )
            matches.push_back( matches_knn[i][0] );
    }

    if (matches.size() <= 20) //匹配点太少
        return false;

    for ( auto m:matches )
    {
        points1.push_back( kp1[m.queryIdx].pt );
        points2.push_back( kp2[m.trainIdx].pt );
    }

    return true;
}



Overwriting main.cpp


In [ ]:
!rm -r build
!mkdir build
%cd build/

/content/BA_Example/g2o_ba_example/build


In [ ]:
!cmake ..

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found version "4.5.4")
-- Found Eigen3: /usr/local/include/eigen3 (Required is at least version "2.91.0")
eigen3 Directory Found.
-- Configuring done (0.5s)
-- Generating done (0.0s)
-- Build files have been written to: /content/BA_Example/g2o_ba_example/build


In [ ]:
!make

[ 50%] Building CXX object CMakeFiles/ba_example.dir/main.cpp.o
[100%] Linking CXX executable ba_example
[100%] Built target ba_example


In [ ]:
# Call format: command [first image] [second image]
!./ba_example

分别找到了500和500个特征点
找到了336组对应特征点。
开始优化
iteration= 0	 chi2= 1144.656708	 time= 0.00114542	 cumTime= 0.00114542	 edges= 672	 schur= 1	 lambda= 11.541631	 levenbergIter= 1
iteration= 1	 chi2= 798.545656	 time= 0.000357314	 cumTime= 0.00150273	 edges= 672	 schur= 1	 lambda= 7.694421	 levenbergIter= 1
iteration= 2	 chi2= 583.408122	 time= 0.000340705	 cumTime= 0.00184344	 edges= 672	 schur= 1	 lambda= 5.129614	 levenbergIter= 1
iteration= 3	 chi2= 562.648106	 time= 0.000320372	 cumTime= 0.00216381	 edges= 672	 schur= 1	 lambda= 3.419743	 levenbergIter= 1
iteration= 4	 chi2= 522.257066	 time= 0.000323048	 cumTime= 0.00248685	 edges= 672	 schur= 1	 lambda= 2.279828	 levenbergIter= 1
iteration= 5	 chi2= 508.204465	 time= 0.000428751	 cumTime= 0.00291561	 edges= 672	 schur= 1	 lambda= 3.039771	 levenbergIter= 2
iteration= 6	 chi2= 469.357119	 time= 0.000301688	 cumTime= 0.00321729	 edges= 672	 schur= 1	 lambda= 1.013257	 levenbergIter= 1
iteration= 7	 chi2= 464.272510	 time= 0.000456939	 cumTime= 